# Gemini RAG Pipeline Demo

This notebook uses your existing files:
- `chunking.py`
- `embeding.py`
- `main.py`

Run cells top-to-bottom.

In [1]:
from pathlib import Path
import os

from chunking import read_text_files, chunk_documents
from embeding import GeminiClient
from main import load_environment, build_index, answer_question

print("Imports successful.")

Imports successful.


In [2]:
# Config (you can change these)
DOCS_DIR = Path("sample_docs")
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
TOP_K = 3

DOCS_DIR.mkdir(exist_ok=True)
print("Docs directory:", DOCS_DIR.resolve())

Docs directory: C:\Users\gkc\Documents\docker_learn\docker_masterclass\src\sample_docs


In [3]:
# Optional: create sample docs if folder is empty
has_docs = any(DOCS_DIR.rglob("*.txt")) or any(DOCS_DIR.rglob("*.md"))
if not has_docs:
    (DOCS_DIR / "rag_intro.txt").write_text(
        "RAG stands for Retrieval Augmented Generation. "
        "It retrieves relevant chunks first, then uses them to answer. "
        "Embeddings convert text into vectors for semantic search.",
        encoding="utf-8",
    )
    (DOCS_DIR / "docker_note.md").write_text(
        "Docker packages app code and dependencies into containers. "
        "Containers help keep environments consistent across machines.",
        encoding="utf-8",
    )
    print("Created sample docs in", DOCS_DIR)
else:
    print("Using existing docs from", DOCS_DIR)

Using existing docs from sample_docs


In [4]:
documents = read_text_files(DOCS_DIR)
print(f"Read {len(documents)} documents from {DOCS_DIR}")

Read 2 documents from sample_docs


In [5]:
chunks = chunk_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
# print(f"Created {len(chunks)} chunks")
chunks[:10]  # Limit to first 10 chunks for demo

[TextChunk(chunk_id='docker_note_chunk_0', source='sample_docs\\docker_note.md', text='Docker packages app code and dependencies into containers. Containers help keep environments consistent across machines.'),
 TextChunk(chunk_id='rag_intro_chunk_0', source='sample_docs\\rag_intro.txt', text='RAG stands for Retrieval Augmented Generation. It retrieves relevant chunks first, then uses them to answer. Embeddings convert text into vectors for semantic search.')]

In [6]:
# Step 1: load and chunk documents

chunks = chunk_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print("Document count:", len(documents))
print("Chunk count:", len(chunks))
if chunks:
    print("\nFirst chunk preview:\n")
    print("chunk_id:", chunks[0].chunk_id)
    print("source:", chunks[0].source)
    print("text:", chunks[0].text[:300], "...")

Document count: 2
Chunk count: 2

First chunk preview:

chunk_id: docker_note_chunk_0
source: sample_docs\docker_note.md
text: Docker packages app code and dependencies into containers. Containers help keep environments consistent across machines. ...


In [8]:
# Step 2: load API key from .env and create Gemini client
load_environment()

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError("Missing API key. Add GEMINI_API_KEY to your .env file.")

embedding_model = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-001")
generation_model = os.getenv("GEMINI_GENERATION_MODEL", "gemini-pro")

client = GeminiClient(
    api_key=api_key,
    embedding_model=embedding_model,
    generation_model=generation_model,
)

print("Gemini client ready.")
print("Embedding model:", embedding_model)
print("Generation model:", generation_model)

Gemini client ready.
Embedding model: gemini-embedding-001
Generation model: gemini-2.5-flash


In [9]:
# Step 3: build vector index (embeds every chunk)
store, indexed_chunks = build_index(
    client=client,
    docs_dir=DOCS_DIR,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

print("\nIndex ready. Indexed chunks:", len(indexed_chunks))

Indexed chunk 1/2
Indexed chunk 2/2

Index ready. Indexed chunks: 2


In [10]:
# Step 4: ask a question
question = "What is RAG and why are embeddings used?"
answer, matches = answer_question(
    client=client,
    store=store,
    question=question,
    top_k=TOP_K,
)

print("Question:\n", question)
print("\nAnswer:\n", answer)
print("\nTop sources:")
for m in matches:
    print(f"- {m.source} | score={m.score:.4f}")

Question:
 What is RAG and why are embeddings used?

Answer:
 RAG stands for Retrieval Augmented Generation. It works by first retrieving relevant chunks of information and then using those chunks to formulate an answer.

Embeddings are used to convert text into vectors, which enables semantic search.

Top sources:
- sample_docs\rag_intro.txt | score=0.8569
- sample_docs\docker_note.md | score=0.5706


In [ ]:
# Optional interactive loop (stop with exit)
while True:
    user_q = input("You: ").strip()
    if user_q.lower() in {"exit", "quit"}:
        print("Bye")
        break
    if not user_q:
        continue
    ans, top = answer_question(client=client, store=store, question=user_q, top_k=TOP_K)
    print("\nAssistant:\n", ans)
    print("\nTop sources:")
    for m in top:
        print(f"- {m.source} | score={m.score:.4f}")
    print()